# Penanganan outlier: potong, buang, atau biarkan
**unsupervised › preprocessing** · notebook `outlier_treatment`

**Kapan dipakai:** Ada nilai ekstrem yang mungkin galat pengukuran dan merusak model linear.

**Jebakan umum:** Outlier bisa jadi sinyal sah (mis. transaksi besar). Jangan membuang tanpa alasan domain; validasi pada data yang TIDAK dibersihkan.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi

In [ ]:
PATH = ROOT / "data_latihan/tabular/regression/diamonds_train.csv"; LABEL_COL = "price"
MODEL_KEY = "outlier_treatment"
OUT = out_dir("unsupervised", "preprocessing", MODEL_KEY)
SEED = 42; set_seed(SEED)

## 2. Data: harga berlian. Kolom `x`, `y`, `z` memuat nilai 0 (tidak mungkin) dan outlier ekstrem (galat pengukuran klasik)

In [ ]:
from sklearn.model_selection import train_test_split
df = pd.read_csv(PATH); feats = ["carat", "depth", "table", "x", "y", "z"]
Xtr, Xva, ytr, yva = train_test_split(df[feats], df[LABEL_COL].to_numpy(float), test_size=0.25, random_state=SEED)
display(Xtr.describe().loc[["min", "50%", "max"]].round(2)); print("baris dengan x/y/z = 0:", int(((Xtr[["x", "y", "z"]] == 0).any(axis=1)).sum()))

## 3. Empat perlakuan, diterapkan HANYA pada training; validation dibiarkan apa adanya
Validation tidak boleh dibersihkan: data nyata di masa depan juga akan memiliki outlier.

In [ ]:
from sklearn.linear_model import LinearRegression, HuberRegressor
from sklearn.ensemble import IsolationForest
from sklearn.metrics import mean_squared_error, mean_absolute_error
def iqr_clip(A, B):                                        # winsorize: potong ke [Q1-1.5IQR, Q3+1.5IQR] dihitung dari train
    q1, q3 = A.quantile(.25), A.quantile(.75); lo, hi = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1); return A.clip(lo, hi, axis=1), B.clip(lo, hi, axis=1)
treat = {}
treat["tanpa perlakuan"] = (Xtr, ytr)
a, _ = iqr_clip(Xtr, Xva); treat["potong IQR (winsorize)"] = (a, ytr)
z = ((Xtr - Xtr.mean()) / Xtr.std()).abs().max(axis=1); treat["buang |z|>4"] = (Xtr[z <= 4], ytr[(z <= 4).to_numpy()])
iso = IsolationForest(contamination=0.03, random_state=SEED).fit(Xtr); keep = iso.predict(Xtr) == 1; treat["buang 3% terjanggal (iForest)"] = (Xtr[keep], ytr[keep])
rows = []
for name, (A, ya) in treat.items():
    for mname, mdl in (("linear", LinearRegression()), ("huber", HuberRegressor(max_iter=500))):
        B = iqr_clip(Xtr, Xva)[1] if name.startswith("potong") else Xva              # potong juga validation memakai batas dari train
        pr = mdl.fit(A, ya).predict(B)
        rows.append({"perlakuan": name, "model": mname, "n_latih": len(A), "RMSE": float(np.sqrt(mean_squared_error(yva, pr))), "MAE": mean_absolute_error(yva, pr)})
res = pd.DataFrame(rows); display(res.pivot(index="perlakuan", columns="model", values="RMSE").round(1))

Baca hasilnya apa adanya: pada validation yang **tetap berisi outlier**, perlakuan pada training tidak otomatis memperbaiki galat, dan model robust (Huber) juga tidak otomatis lebih baik. Efek bergantung data dan metrik (RMSE sangat dipengaruhi nilai ekstrem). Putuskan berdasarkan validasi dan alasan domain: **apakah nilai ekstrem itu galat (mis. dimensi 0 mm) atau sinyal sah?** Untuk galat yang jelas salah (x/y/z = 0), memperbaikinya atau membuangnya lebih beralasan daripada memotong semua ekor distribusi.

In [ ]:
save_json({"results": res.to_dict("records")}, OUT / "metrics.json"); print("OK. hasil di", OUT)

## Langkah berikutnya
- Model pohon/boosting jarang butuh scaling atau pembersihan outlier; fokus di sana ke fitur dan validasi.
- Selalu ukur efek perlakuan pada validation yang tidak diperlakukan.